# 08 -- Дополнительные признаки и проверка данных

Проверяю конкретные причины ошибок: широкие заголовки, редкие слова, условия в фильтрах,
числа в запросе и несогласие lexical/dense. Новые признаки строю без целевых меток.
Сравнение моделей будет в 09, поэтому здесь не выбираю признаки по dev или test.

In [1]:
from pathlib import Path
import sys
import json
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

fx_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
get_ipython().run_line_magic('run', str(fx_root / 'notebooks/common.ipynb'))

fx_out = fx_root / 'outputs'
fx_seed = 143
np.random.seed(fx_seed)


## 1. Каталог и веса редких слов

IDF считаю по заголовкам известного каталога. Выборы пользователей здесь не нужны.
Редкое слово «видеодомофон» полезнее общего «услуги»; взвешенное покрытие различает такие совпадения.

In [2]:
fx_items = pd.read_parquet(fx_out / 'cache/items.parquet')
fx_queries = pd.read_parquet(fx_out / 'cache/eval_queries.parquet')
fx_df = Counter()
for fx_title in fx_items.title_text:
    fx_df.update(set(fx_title.split()))
fx_idf = {
    word: float(np.log1p(len(fx_items) / (1 + count))) for word, count in fx_df.items()
}
write_json(fx_idf, fx_out / 'models/title_idf.json')


## 2. Дополнение EDA

Проверяю, сколько заголовков повторяются у разных объявлений, насколько длинные запросы,
как часто встречаются числа и заполнены фильтры. Текстовый дубль не удаляю: разные исполнители
могут одинаково назвать услугу. Пропуски и хвосты исходных полей уже разобраны в 00.

In [3]:
fx_summary = pd.Series(
    {
        'catalog_items': len(fx_items),
        'unique_titles': fx_items.title_text.nunique(),
        'items_with_nonunique_title_share': fx_items.title_text.duplicated(
            keep=False
        ).mean(),
        'empty_title_share': fx_items.title_text.eq('').mean(),
        'eval_query_has_number_share': fx_queries.text_key.str.contains(
            r'\d', regex=True
        ).mean(),
        'eval_nonempty_filters_share': fx_queries.search_infm_params_text.map(normalize)
        .ne('')
        .mean(),
    }
)
fx_summary.to_csv(fx_out / 'validation/extended_eda.csv', header=['value'])
display(fx_summary)
fx_query_lengths = fx_queries.text_key.str.split().str.len()
display(fx_query_lengths.describe(percentiles=[0.5, 0.9, 0.99]))


catalog_items                       515895.000000
unique_titles                       273561.000000
items_with_nonunique_title_share         0.547381
empty_title_share                        0.000000
eval_query_has_number_share              0.031071
eval_nonempty_filters_share              0.610000
dtype: float64

count    8400.000000
mean        2.764048
std         1.117373
min         1.000000
50%         3.000000
90%         4.000000
99%         6.000000
max         9.000000
Name: text_key, dtype: float64

## 3. Признаки пары

- Jaccard и precision заголовка отличают точное совпадение от длинного перечня услуг.
- Exact phrase проверяет порядок слов. IDF coverage усиливает редкие слова.
- Покрытие чисел помогает запросам с моделью техники или размером; не считаю любое число ценой.
- Покрытие фильтров параметрами -- мягкий сигнал: текст фильтров может содержать служебные слова.
- Логарифм расстояния и взаимодействия с локальностью описывают географию без жесткого запрета.
- Согласие источников и положение внутри пула показывают уверенность поиска.

Признаки относительно пула вычисляю отдельно для каждого запроса, без target.
Их распределение зависит от генераторов и размера пула; везде сохраняю прежние 400 кандидатов.

In [4]:
fx_feature_rows = []
for fx_split in ['rank_train', 'dev', 'test']:
    fx_frame = pd.read_parquet(fx_out / f'cache/fusion_{fx_split}.parquet')
    fx_extended = extra_features(fx_frame, fx_queries, fx_items, fx_idf)
    assert len(fx_extended) == len(fx_frame)
    assert fx_extended[['query_id', 'pos']].equals(fx_frame[['query_id', 'pos']])
    assert not {'target', 'split', 'regime'} & set(fx_extended.columns)
    fx_extended.to_parquet(fx_out / f'cache/extended_{fx_split}.parquet', index=False)
    fx_feature_rows.append(
        {
            'split': fx_split,
            'rows': len(fx_extended),
            'new_features': len(fx_extended.columns) - len(fx_frame.columns),
        }
    )
display(pd.DataFrame(fx_feature_rows))


,split,rows,new_features
0,rank_train,1731281,21
1,dev,784839,21
2,test,785570,21


## 4. Пропуски новых признаков

Пустой фильтр и запрос без чисел дают нулевое покрытие вместе с индикаторами наличия.
Неизвестную цену и расстояние оставляю пропуском до подготовки матрицы модели.
Проверяю конечность непустых значений; положительную разметку здесь не загружаю.

In [5]:
fx_new_columns = [col for col in fx_extended if col not in fx_frame]
fx_audit = fx_extended[fx_new_columns].agg(['min', 'max']).T
fx_audit['missing_share'] = fx_extended[fx_new_columns].isna().mean()
assert not np.isinf(fx_extended[fx_new_columns].to_numpy()).any()
fx_audit.to_csv(fx_out / 'validation/extended_feature_audit.csv')
display(fx_audit)


,min,max,missing_share
title_jaccard,0.000000,1.000000,0.000000
title_precision,0.000000,1.000000,0.000000
title_idf_coverage,0.000000,1.000000,0.000000
title_exact_phrase,0.000000,1.000000,0.000000
title_extra_tokens,0.000000,17.000000,0.000000
filter_param_coverage,0.000000,1.000000,0.000000
query_has_numbers,0.000000,1.000000,0.000000
number_coverage,0.000000,1.000000,0.000000
distance_log1p,0.000000,9.559117,0.120166
remote_distance,0.000000,9.559117,0.120166
